# Module 0: Python & NumPy Foundations

**Prerequisites:** Basic Python (functions, classes, list comprehensions). No prior NumPy experience required.

**Learning objectives:** Master NumPy internals, broadcasting, advanced indexing, einsum, vectorization, and random number generation.

**Estimated time:** 2–3 hours

**No papers for this module** — this is foundational tooling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import Tuple

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)
print(f"NumPy version: {np.__version__}")

---
## 0.1 — Array Internals: dtype, strides, contiguous vs non-contiguous

Every NumPy array is a thin wrapper around a flat buffer of bytes. Three pieces of metadata make it behave like a multidimensional array:

| Attribute | What it stores | Why it matters |
|-----------|---------------|----------------|
| **dtype** | Element type and byte-width (e.g. `float32` = 4 bytes) | Determines precision, memory, and SIMD eligibility |
| **shape** | Tuple of dimension sizes | Defines the logical layout |
| **strides** | Bytes to jump per axis to reach the next element | Enables views (transpose, slice) without copying data |

### C-contiguous vs F-contiguous

- **C-contiguous (row-major):** last axis varies fastest in memory. Default in NumPy.
- **F-contiguous (column-major):** first axis varies fastest. Default in Fortran/MATLAB.

A transposed C-contiguous array is F-contiguous (and vice versa) — no data is moved, only strides change.

### Views vs Copies

- A **view** shares the underlying buffer. Mutations propagate.
- A **copy** allocates new memory. Independent of the original.
- Rule of thumb: slicing and reshaping (when possible) return views; fancy indexing returns copies.

In [ ]:
# --- Worked Example: Inspecting array internals ---

a = np.arange(12, dtype=np.float32).reshape(3, 4)  # shape (3, 4)
print("Array a:")
print(a)
print(f"  dtype   : {a.dtype}")
print(f"  shape   : {a.shape}")
print(f"  strides : {a.strides}  (bytes per axis)")
print(f"  itemsize: {a.itemsize} bytes")
print(f"  C-contig: {a.flags['C_CONTIGUOUS']}")
print(f"  F-contig: {a.flags['F_CONTIGUOUS']}")
print()

# Strides explanation: stride[0]=16 means skip 16 bytes (4 floats * 4 bytes) to move one row down.
# stride[1]=4 means skip 4 bytes (1 float) to move one column right.

# Transpose: strides swap, no data copy
b = a.T  # shape (4, 3)
print("Transposed b = a.T:")
print(f"  shape   : {b.shape}")
print(f"  strides : {b.strides}  (swapped!)")
print(f"  C-contig: {b.flags['C_CONTIGUOUS']}")
print(f"  F-contig: {b.flags['F_CONTIGUOUS']}")
print(f"  Shares memory with a? {np.shares_memory(a, b)}")

In [ ]:
# --- View vs Copy demo ---

# Slicing returns a VIEW
view_of_a = a[0:2, 1:3]  # shape (2, 2)
print(f"view_of_a shares memory with a? {np.shares_memory(view_of_a, a)}")

view_of_a[0, 0] = -999.0
print(f"After mutating view, a[0,1] = {a[0, 1]}  (mutated through view!)")
a[0, 1] = 1.0  # restore

# Fancy indexing returns a COPY
copy_of_a = a[[0, 2], :]  # shape (2, 4)
print(f"\ncopy_of_a shares memory with a? {np.shares_memory(copy_of_a, a)}")

copy_of_a[0, 0] = -999.0
print(f"After mutating copy, a[0,0] = {a[0, 0]}  (unchanged!)")

In [ ]:
# --- dtype performance difference ---

size = 5_000_000
x64 = rng.standard_normal(size)               # float64 by default
x32 = x64.astype(np.float32)                  # float32

t0 = time.perf_counter()
_ = x64 @ x64
t64 = time.perf_counter() - t0

t0 = time.perf_counter()
_ = x32 @ x32
t32 = time.perf_counter() - t0

print(f"float64 dot: {t64*1000:.2f} ms  |  float32 dot: {t32*1000:.2f} ms")
print(f"Memory: float64 = {x64.nbytes / 1e6:.1f} MB, float32 = {x32.nbytes / 1e6:.1f} MB")

### Exercise 0.1

1. Create a `(6, 8)` float64 array `m`. Predict its strides, then verify.
2. Take a slice `s = m[::2, ::3]`. Is `s` a view or copy? What are its strides? Verify.
3. Call `s_flat = s.reshape(-1)`. Does this return a view or copy? Why?
4. Write a function `force_c_contiguous(arr)` that returns `arr` unchanged if already C-contiguous, otherwise returns a C-contiguous copy.

In [ ]:
# YOUR CODE HERE — Exercise 0.1

# 1. Create a (6, 8) float64 array m. Predict its strides, then verify.
m = np.arange(48, dtype=np.float64).reshape(6, 8)
# Prediction: stride[0] = ?, stride[1] = ?
# print(f"m.strides = {m.strides}")

# 2. Take a slice s = m[::2, ::3]. View or copy? What strides?

# 3. Call s.reshape(-1). View or copy? Why?

# 4. Write force_c_contiguous(arr) that returns arr if C-contiguous,
#    else returns a C-contiguous copy.
def force_c_contiguous(arr: np.ndarray) -> np.ndarray:
    """Return arr if C-contiguous, else a C-contiguous copy."""
    pass


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# 1. Strides prediction: float64 = 8 bytes, shape (6,8)
#    stride[0] = 8 * 8 = 64, stride[1] = 8
m = np.arange(48, dtype=np.float64).reshape(6, 8)  # shape (6, 8)
print(f"m.strides = {m.strides}")  # (64, 8) -- confirmed

# 2. Slicing with steps returns a VIEW with scaled strides
s = m[::2, ::3]  # shape (3, 3) -- rows 0,2,4; cols 0,3,6
print(f"s is a view: {np.shares_memory(s, m)}")
print(f"s.strides = {s.strides}")  # (128, 24) = (64*2, 8*3)
print(f"s.shape   = {s.shape}")

# 3. reshape(-1) on non-contiguous array requires a COPY
s_flat = s.reshape(-1)  # shape (9,)
print(f"\ns_flat shares memory with m? {np.shares_memory(s_flat, m)}")  # False -- copy!
# Because s has non-unit strides on both axes, the elements are not
# contiguous in memory, so reshape must copy.


# 4.
def force_c_contiguous(arr: np.ndarray) -> np.ndarray:
    """Return arr if C-contiguous, else a C-contiguous copy."""
    if arr.flags['C_CONTIGUOUS']:
        return arr
    return np.ascontiguousarray(arr)


# Test
t = m.T  # F-contiguous, not C-contiguous
t_c = force_c_contiguous(t)
print(f"\nt is C-contig: {t.flags['C_CONTIGUOUS']}")
print(f"t_c is C-contig: {t_c.flags['C_CONTIGUOUS']}")
print(f"t_c shares memory with m: {np.shares_memory(t_c, m)}")
assert np.array_equal(t, t_c)

---
## 0.2 — Broadcasting: The Rules, Common Pitfalls, Mental Model

**Why this matters for diffusion:** Nearly every diffusion operation involves arrays of different shapes — adding noise vectors to image batches, scaling by per-timestep coefficients, normalizing across channels. Broadcasting lets you write these as clean one-liners instead of manual loops or explicit `np.tile`.

### The Three Rules

Broadcasting aligns shapes from the **right** (trailing dimensions) and applies these rules:

$$
\text{For each dimension } d: \quad
\begin{cases}
a_d = b_d & \text{compatible (same size)} \\
a_d = 1 \text{ or } b_d = 1 & \text{compatible (stretch the size-1 dim)} \\
\text{otherwise} & \text{incompatible → error}
\end{cases}
$$

| Rule | Description |
|------|-------------|
| **1. Right-align** | Shapes are compared element-wise starting from the trailing dimension |
| **2. Size-1 expansion** | A dimension of size 1 is stretched to match the other array's size |
| **3. Compatibility** | Dimensions must be equal or one of them must be 1; otherwise, error |

### Mental model

Think of broadcasting as "virtual replication" — NumPy never actually copies the data. Internally, it sets the stride for size-1 dimensions to **0**, so the same value is reused across that axis.

### Common pattern: `keepdims=True`

When computing statistics (mean, std) for normalization, use `keepdims=True` to preserve the reduced axis as size-1. This makes the result directly broadcastable with the original array.

In [ ]:
# --- Worked Example 1: (3,4) + (4,) ---
A = np.ones((3, 4))    # shape (3, 4)
b = np.array([10, 20, 30, 40])  # shape (4,)

# Right-align:  (3, 4)
#                  (4,)  -> treated as (1, 4) -> broadcast to (3, 4)
result = A + b  # shape (3, 4)
print("(3,4) + (4,):")
print(result)
print(f"Result shape: {result.shape}")

In [ ]:
# --- Worked Example 2: Outer sum via broadcasting ---
x = np.array([1, 2, 3])        # shape (3,)
y = np.array([10, 20, 30, 40]) # shape (4,)

# To get outer sum, reshape x to (3,1) and let broadcasting do (3,1)+(1,4)->(3,4)
outer_sum = x[:, None] + y[None, :]  # shape (3, 4)
print("Outer sum (3,1) + (1,4) -> (3,4):")
print(outer_sum)

In [ ]:
# --- Worked Example 3: Subtle broadcasting bug ---

# Suppose we want to subtract per-row means from a (3,4) matrix.
M = rng.standard_normal((3, 4))  # shape (3, 4)
row_means = M.mean(axis=1)       # shape (3,) -- this is WRONG for direct subtraction!

# M.shape = (3, 4), row_means.shape = (3,)
# Right-align: last dims are 4 vs 3 -> INCOMPATIBLE
try:
    bad = M - row_means
except ValueError as e:
    print(f"Error: {e}")

# Fix: keepdims=True preserves the axis as size-1
row_means_correct = M.mean(axis=1, keepdims=True)  # shape (3, 1)
centered = M - row_means_correct  # shape (3, 4)
print(f"\nCentered row means (should be ~0): {centered.mean(axis=1)}")
print(f"np.broadcast_shapes((3,4), (3,1)) = {np.broadcast_shapes((3,4), (3,1))}")

### Exercise 0.2

1. **Predict the output shape** (or "error") for each pair:
   - `(5, 3) + (3,)`
   - `(5, 3) + (5,)`
   - `(2, 1, 4) + (3, 4)`
   - `(2, 3, 4) + (2, 1, 1)`
   - `(7,) + (1, 7, 1)`

2. **Normalize batch of images per-channel.** Given `images` of shape `(B, H, W, C)` and per-channel `mean` and `std` each of shape `(C,)`, normalize so each channel has zero mean and unit variance.

3. **Pairwise Euclidean distances.** Given `X` of shape `(N, D)` and `Y` of shape `(M, D)`, compute the `(N, M)` distance matrix using broadcasting (no loops).

4. **Batched outer product.** Given `u` of shape `(B, M)` and `v` of shape `(B, N)`, compute `out[b, i, j] = u[b, i] * v[b, j]` with shape `(B, M, N)` using broadcasting.

5. **Verify with `np.broadcast_shapes`.** Confirm your shape predictions from part 1.

In [ ]:
# YOUR CODE HERE — Exercise 0.2

# 1. Predict shapes (or "error") — fill in your predictions
# (5, 3) + (3,)      →  ?
# (5, 3) + (5,)      →  ?
# (2, 1, 4) + (3, 4) →  ?
# (2, 3, 4) + (2, 1, 1) →  ?
# (7,) + (1, 7, 1)   →  ?

# 2. Normalize images (B, H, W, C) per-channel given mean & std of shape (C,)
B, H, W, C = 8, 32, 32, 3
images = rng.standard_normal((B, H, W, C)).astype(np.float32) * 50 + 128
mean = np.array([123.68, 116.78, 103.94], dtype=np.float32)
std = np.array([58.39, 57.12, 57.38], dtype=np.float32)
# normalized = ???

# 3. Pairwise Euclidean distances: X (N, D), Y (M, D) → (N, M)
N, M_pw, D = 100, 80, 5
X = rng.standard_normal((N, D))
Y = rng.standard_normal((M_pw, D))
# dist = ???

# 4. Batched outer product: u (B, M), v (B, N) → (B, M, N)
B_op, M_op, N_op = 16, 5, 7
u = rng.standard_normal((B_op, M_op))
v = rng.standard_normal((B_op, N_op))
# outer = ???


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# 1. Shape predictions
print("Shape predictions:")
print(f"  (5,3) + (3,)     -> {np.broadcast_shapes((5,3), (3,))}")
try:
    np.broadcast_shapes((5,3), (5,))
except ValueError as e:
    print(f"  (5,3) + (5,)     -> ERROR: {e}")
print(f"  (2,1,4) + (3,4)  -> {np.broadcast_shapes((2,1,4), (3,4))}")
print(f"  (2,3,4) + (2,1,1)-> {np.broadcast_shapes((2,3,4), (2,1,1))}")
print(f"  (7,) + (1,7,1)   -> {np.broadcast_shapes((7,), (1,7,1))}")

# 2. Normalize batch images per-channel
B, H, W, C = 8, 32, 32, 3
images = rng.standard_normal((B, H, W, C)).astype(np.float32) * 50 + 128  # shape (B, H, W, C)
mean = np.array([123.68, 116.78, 103.94], dtype=np.float32)  # shape (C,) = (3,)
std = np.array([58.39, 57.12, 57.38], dtype=np.float32)      # shape (C,) = (3,)

# Broadcasting: (B,H,W,C) - (C,) -> (C,) is right-aligned with last dim
normalized = (images - mean) / std  # shape (B, H, W, C)
print(f"\nNormalized images shape: {normalized.shape}")
print(f"Channel means (should be ~0): {normalized.mean(axis=(0,1,2))}")

# 3. Pairwise Euclidean distances
N, M_pw, D = 100, 80, 5
X = rng.standard_normal((N, D))    # shape (N, D)
Y = rng.standard_normal((M_pw, D)) # shape (M, D)

# Expand: X -> (N,1,D), Y -> (1,M,D), diff -> (N,M,D), then norm over D
diff = X[:, None, :] - Y[None, :, :]  # shape (N, M, D)
dist = np.sqrt((diff ** 2).sum(axis=-1))  # shape (N, M)
print(f"\nPairwise distance matrix shape: {dist.shape}")
print(f"dist[0, 0] = {dist[0, 0]:.4f}")

# Verify one entry manually
manual = np.sqrt(np.sum((X[0] - Y[0]) ** 2))
assert np.isclose(dist[0, 0], manual), "Mismatch!"

# 4. Batched outer product
B_op, M_op, N_op = 16, 5, 7
u = rng.standard_normal((B_op, M_op))  # shape (B, M)
v = rng.standard_normal((B_op, N_op))  # shape (B, N)

# u -> (B, M, 1), v -> (B, 1, N), product -> (B, M, N)
outer = u[:, :, None] * v[:, None, :]  # shape (B, M, N)
print(f"\nBatched outer product shape: {outer.shape}")
assert outer.shape == (B_op, M_op, N_op)

# Verify one entry
assert np.isclose(outer[3, 2, 4], u[3, 2] * v[3, 4])

# 5. Already verified above with np.broadcast_shapes
print("\nAll assertions passed.")

---
## 0.3 — Advanced Indexing: Fancy Indexing, Boolean Masks, `np.where`

**Why this matters for diffusion:** The forward process needs to *gather* schedule values at specific timesteps — given a batch of random $t$ values, index into arrays like $\bar{\alpha}_t$. This is fancy indexing. Boolean masks are essential for thresholding, clipping, and conditional operations during sampling.

### The key distinction

| Type | Syntax example | Returns | Memory |
|------|---------------|--------|--------|
| **Basic slicing** | `a[2:5]`, `a[::2]` | **View** (shared memory) | Zero copy |
| **Fancy (integer array)** | `a[[0, 2, 4]]` | Always a **copy** | New allocation |
| **Boolean mask** | `a[a > 0]` | Always a **copy** (1-D result) | New allocation |
| **Combined** | `a[mask, :3]` | Copy | New allocation |

### Key functions

- `np.where(cond, x, y)` — element-wise ternary (vectorized if/else)
- `np.take(a, indices, axis)` — like fancy indexing along one axis
- `np.take_along_axis(a, indices, axis)` — gather using per-element index arrays (critical for `argsort` results)

### The gather pattern (you'll see this constantly)

```python
# Given schedule of shape (T,) and batch of timesteps of shape (B,)
values_at_t = schedule[timesteps]  # shape (B,) — fancy indexing!
```

In [ ]:
# --- Worked Example: 2D fancy indexing ---

grid = np.arange(20).reshape(4, 5)  # shape (4, 5)
print("Grid:")
print(grid)

# Select elements at (0,1), (2,3), (3,4)
row_idx = np.array([0, 2, 3])
col_idx = np.array([1, 3, 4])
selected = grid[row_idx, col_idx]  # shape (3,)
print(f"\nSelected grid[{list(row_idx)}, {list(col_idx)}] = {selected}")

# Boolean mask: replace negatives with 0
data = rng.standard_normal((3, 4))  # shape (3, 4)
print(f"\nOriginal data:\n{data}")
data_clipped = np.where(data > 0, data, 0.0)  # shape (3, 4)
print(f"Clipped (ReLU):\n{data_clipped}")

In [ ]:
# --- Worked Example: Gather operation (diffusion timestep preview) ---
# In diffusion models, we often need to index into a 1-D schedule
# using a batch of timesteps.

T = 1000  # total timesteps
schedule = np.linspace(0.0001, 0.02, T)  # shape (T,) -- beta schedule

# A batch of random timesteps
batch_t = rng.integers(0, T, size=(8,))  # shape (8,)
print(f"Batch timesteps: {batch_t}")

# Gather: index schedule with batch_t
beta_t = schedule[batch_t]  # shape (8,)
print(f"Corresponding betas: {beta_t}")
print(f"Same as np.take: {np.array_equal(beta_t, np.take(schedule, batch_t))}")

### Exercise 0.3

1. **Top-k without full sort.** Given a 1-D array `scores` of shape `(N,)`, find the indices of the top-5 values using `np.argpartition` (O(N)) instead of `np.argsort` (O(N log N)). Return them sorted in descending order.

2. **Batch crop extraction.** Given `images` of shape `(B, H, W)` and per-image crop coordinates `rows` and `cols` each of shape `(B,)`, extract the `(B,)` array of pixel values at those coordinates.

3. **np.where from scratch.** Implement `my_where(cond, x, y)` using only boolean indexing and array allocation (no `np.where`). Verify it matches `np.where` on a test case.

In [ ]:
# YOUR CODE HERE — Exercise 0.3

# 1. Top-k without full sort. Use np.argpartition (O(N)) instead of
#    np.argsort (O(N log N)). Return top-5 indices in descending order.
def topk_indices(scores: np.ndarray, k: int = 5) -> np.ndarray:
    """Return indices of top-k values in descending order."""
    pass

# 2. Batch crop extraction. Given images (B, H, W) and per-image
#    coordinates rows (B,), cols (B,), extract pixels at those coords.
#    Hint: use np.arange for the batch dimension.

# 3. Implement my_where(cond, x, y) using only boolean indexing.
def my_where(cond: np.ndarray, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Element-wise ternary: return x where cond is True, y otherwise."""
    pass


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# 1. Top-k without full sort
def topk_indices(scores: np.ndarray, k: int = 5) -> np.ndarray:
    """Return indices of top-k values in descending order, using O(N) partitioning."""
    # argpartition puts the k largest at the end (indices [-k:])
    partition_idx = np.argpartition(scores, -k)[-k:]  # shape (k,)
    # Sort those k indices by their actual scores (descending)
    top_scores = scores[partition_idx]  # shape (k,)
    sorted_order = np.argsort(top_scores)[::-1]  # shape (k,)
    return partition_idx[sorted_order]  # shape (k,)


scores = rng.standard_normal(1000)  # shape (1000,)
top5 = topk_indices(scores, k=5)
print(f"Top-5 indices: {top5}")
print(f"Top-5 values:  {scores[top5]}")

# Verify against full sort
full_sort_top5 = np.argsort(scores)[::-1][:5]
assert np.array_equal(top5, full_sort_top5), "Mismatch!"
print("Verified against full sort.")

# 2. Batch crop extraction
B_img, H_img, W_img = 16, 64, 64
images_batch = rng.standard_normal((B_img, H_img, W_img))  # shape (B, H, W)
rows = rng.integers(0, H_img, size=(B_img,))  # shape (B,)
cols = rng.integers(0, W_img, size=(B_img,))  # shape (B,)

# Use fancy indexing with arange for the batch dimension
batch_idx = np.arange(B_img)  # shape (B,)
pixels = images_batch[batch_idx, rows, cols]  # shape (B,)
print(f"\nExtracted pixels shape: {pixels.shape}")

# Verify
for i in range(B_img):
    assert pixels[i] == images_batch[i, rows[i], cols[i]]
print("All pixel extractions verified.")

# 3. np.where from scratch
def my_where(cond: np.ndarray, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Element-wise ternary: return x where cond is True, y otherwise."""
    cond = np.asarray(cond, dtype=bool)
    out = np.empty_like(x)
    out[cond] = x[cond]
    out[~cond] = y[~cond]
    return out


test_cond = rng.standard_normal((5, 5)) > 0
test_x = np.ones((5, 5)) * 100
test_y = np.ones((5, 5)) * -100
assert np.array_equal(my_where(test_cond, test_x, test_y),
                      np.where(test_cond, test_x, test_y))
print("\nmy_where matches np.where.")

---
## 0.4 — Einsum Mastery

**Why this matters for diffusion:** Attention mechanisms — the core of modern diffusion architectures (U-Net with attention, DiT) — involve batched matrix multiplies across heads and sequence dimensions. Einsum expresses these operations in a single, readable line.

### The notation

Einstein summation provides a unified syntax for tensor contractions:

$$
C_{ik} = \sum_j A_{ij} B_{jk} \quad \Longleftrightarrow \quad \texttt{np.einsum('ij,jk->ik', A, B)}
$$

**Rules:**
1. Each input gets a subscript string — one letter per axis.
2. The output subscript specifies which indices survive.
3. Any index in the inputs but **not** in the output is **summed over** (contracted).

### Common patterns reference

| Operation | Einsum | Equivalent |
|-----------|--------|-----------|
| Matrix multiply | `ij,jk->ik` | `A @ B` |
| Batch matmul | `bij,bjk->bik` | `np.einsum(...)` or `@` with batch dims |
| Dot product | `i,i->` | `np.dot(a, b)` |
| Outer product | `i,j->ij` | `np.outer(a, b)` |
| Trace | `ii->` | `np.trace(A)` |
| Transpose | `ij->ji` | `A.T` |
| Row sum | `ij->i` | `A.sum(axis=1)` |
| Column sum | `ij->j` | `A.sum(axis=0)` |
| Element-wise × then sum | `ij,ij->` | `(A * B).sum()` |
| **Attention scores** | `bhqd,bhkd->bhqk` | $\text{scores}_{b,h,q,k} = \sum_d Q_{b,h,q,d} K_{b,h,k,d}$ |
| **Bilinear form** $\mathbf{x}^\top A \mathbf{x}$ | `i,ij,j->` | Custom |

In [ ]:
# --- Worked Example: Attention scores ---

B_attn, H, Q, K, D_head = 2, 4, 8, 8, 16
queries = rng.standard_normal((B_attn, H, Q, D_head))  # shape (B, H, Q, D)
keys = rng.standard_normal((B_attn, H, K, D_head))     # shape (B, H, K, D)

# Attention: scores[b,h,q,k] = sum_d queries[b,h,q,d] * keys[b,h,k,d]
scores = np.einsum('bhqd,bhkd->bhqk', queries, keys)  # shape (B, H, Q, K)
print(f"Attention scores shape: {scores.shape}")

# Verify against manual loop for one element
manual_val = np.sum(queries[0, 1, 2, :] * keys[0, 1, 3, :])
assert np.isclose(scores[0, 1, 2, 3], manual_val)
print(f"scores[0,1,2,3] = {scores[0,1,2,3]:.4f} (verified)")

In [ ]:
# --- Worked Example: Batched bilinear form x^T A x ---

B_bi, D_bi = 32, 5
x_bi = rng.standard_normal((B_bi, D_bi))           # shape (B, D)
A_bi = rng.standard_normal((B_bi, D_bi, D_bi))     # shape (B, D, D)

# x^T A x for each batch: contract over both i and j
# result[b] = sum_i sum_j x[b,i] * A[b,i,j] * x[b,j]
result_bi = np.einsum('bi,bij,bj->b', x_bi, A_bi, x_bi)  # shape (B,)
print(f"Bilinear form shape: {result_bi.shape}")

# Verify one element
manual_bi = x_bi[0] @ A_bi[0] @ x_bi[0]
assert np.isclose(result_bi[0], manual_bi)
print(f"result[0] = {result_bi[0]:.4f} (verified)")

### Exercise 0.4

**Part A: 10 translations.** For each operation, write the einsum string AND verify it produces the correct result.

1. Matrix-vector product: `A @ v` where `A` is `(M, N)`, `v` is `(N,)`
2. Element-wise product then sum all: `(A * B).sum()`
3. Batch matrix multiply: `X @ Y` where `X` is `(B, M, N)`, `Y` is `(B, N, P)`
4. Trace of a matrix: `np.trace(A)`
5. Diagonal extraction: `np.diag(A)`
6. Sum over last axis: `A.sum(axis=-1)` for 3-D `A`
7. Hadamard (element-wise) product: `A * B` (keep all dims)
8. Batch dot product: `(u * v).sum(axis=-1)` for `u, v` of shape `(B, D)`
9. Tensor contraction: contract middle axis of `(A, B, C)` with `(B,)` -> `(A, C)`
10. Kronecker-like: outer product of each row of `(M, N)` with itself -> `(M, N, N)`

**Part B: Multi-head attention dot products.** Given `Q` shape `(B, H, S, D)` and `K` shape `(B, H, S, D)`, compute scaled attention logits `Q K^T / sqrt(D)` using einsum.

**Part C: Batch Frobenius norm.** Given `X` shape `(B, M, N)`, compute `||X_b||_F` for each batch using einsum to get `(B,)` of squared norms, then sqrt.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part A: 10 translations

M_a, N_a, P_a, B_a, D_a, C_a = 4, 5, 3, 2, 6, 7
A_sq = rng.standard_normal((N_a, N_a))  # square matrix for trace/diag
A_mn = rng.standard_normal((M_a, N_a))
B_mn = rng.standard_normal((M_a, N_a))
v_n = rng.standard_normal((N_a,))
X_3d = rng.standard_normal((B_a, M_a, N_a))
Y_3d = rng.standard_normal((B_a, N_a, P_a))
u_bd = rng.standard_normal((B_a, D_a))
v_bd = rng.standard_normal((B_a, D_a))
T_abc = rng.standard_normal((M_a, N_a, C_a))
w_b = rng.standard_normal((N_a,))

# 1. Matrix-vector product
r1 = np.einsum('ij,j->i', A_mn, v_n)  # shape (M,)
assert np.allclose(r1, A_mn @ v_n)
print("1. ij,j->i (mat-vec)  OK")

# 2. Element-wise product then sum
r2 = np.einsum('ij,ij->', A_mn, B_mn)  # scalar
assert np.isclose(r2, (A_mn * B_mn).sum())
print("2. ij,ij->  (elem-sum) OK")

# 3. Batch matmul
r3 = np.einsum('bmn,bnp->bmp', X_3d, Y_3d)  # shape (B, M, P)
assert np.allclose(r3, X_3d @ Y_3d)
print("3. bmn,bnp->bmp (batch matmul) OK")

# 4. Trace
r4 = np.einsum('ii->', A_sq)  # scalar
assert np.isclose(r4, np.trace(A_sq))
print("4. ii->  (trace) OK")

# 5. Diagonal
r5 = np.einsum('ii->i', A_sq)  # shape (N,)
assert np.allclose(r5, np.diag(A_sq))
print("5. ii->i (diag) OK")

# 6. Sum over last axis of 3-D
r6 = np.einsum('ijk->ij', X_3d)  # shape (B, M)
assert np.allclose(r6, X_3d.sum(axis=-1))
print("6. ijk->ij (sum last) OK")

# 7. Hadamard product
r7 = np.einsum('ij,ij->ij', A_mn, B_mn)  # shape (M, N)
assert np.allclose(r7, A_mn * B_mn)
print("7. ij,ij->ij (hadamard) OK")

# 8. Batch dot product
r8 = np.einsum('bd,bd->b', u_bd, v_bd)  # shape (B,)
assert np.allclose(r8, (u_bd * v_bd).sum(axis=-1))
print("8. bd,bd->b (batch dot) OK")

# 9. Tensor contraction: contract axis 1 of (A,B,C) with (B,)
r9 = np.einsum('abc,b->ac', T_abc, w_b)  # shape (A, C)
assert np.allclose(r9, np.tensordot(T_abc, w_b, axes=([1], [0])))
print("9. abc,b->ac (contract mid) OK")

# 10. Outer product of each row with itself
r10 = np.einsum('mi,mj->mij', A_mn, A_mn)  # shape (M, N, N)
assert np.allclose(r10[0], np.outer(A_mn[0], A_mn[0]))
print("10. mi,mj->mij (row outer) OK")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part B: Multi-head attention

B_mha, H_mha, S_mha, D_mha = 2, 8, 16, 64
Q_mha = rng.standard_normal((B_mha, H_mha, S_mha, D_mha))  # shape (B, H, S, D)
K_mha = rng.standard_normal((B_mha, H_mha, S_mha, D_mha))  # shape (B, H, S, D)

# Q K^T / sqrt(D): contract over D, keep both sequence dims
logits = np.einsum('bhqd,bhkd->bhqk', Q_mha, K_mha) / np.sqrt(D_mha)  # shape (B, H, S, S)
print(f"Part B: attention logits shape = {logits.shape}")

# Verify
manual_logit = (Q_mha[0, 0] @ K_mha[0, 0].T) / np.sqrt(D_mha)
assert np.allclose(logits[0, 0], manual_logit)
print("Verified against manual matmul.")

# ✅ SOLUTION — try the exercise above before running this -- Part C: Batch Frobenius norm

B_fn, M_fn, N_fn = 10, 4, 6
X_fn = rng.standard_normal((B_fn, M_fn, N_fn))  # shape (B, M, N)

# Frobenius norm squared = sum of squared elements
frob_sq = np.einsum('bmn,bmn->b', X_fn, X_fn)  # shape (B,)
frob_norm = np.sqrt(frob_sq)  # shape (B,)
print(f"\nPart C: Frobenius norms shape = {frob_norm.shape}")

# Verify
expected_frob = np.linalg.norm(X_fn.reshape(B_fn, -1), axis=1)
assert np.allclose(frob_norm, expected_frob)
print(f"Norms: {frob_norm}")
print("Verified against np.linalg.norm.")

---
## 0.5 — Vectorization Patterns: Replacing Loops

**Why this matters for diffusion:** Diffusion training processes millions of images. Every Python loop is a bottleneck — vectorized NumPy operations dispatch to optimized C/Fortran BLAS routines that exploit SIMD, cache locality, and parallelism. The performance gap is typically **10–100×**.

### The core rule

Replace Python `for` loops with NumPy operations. If you're iterating over array elements in Python, you're doing it wrong.

### Key tools for vectorization

| Function | Use case | Diffusion relevance |
|----------|----------|--------------------|
| `np.cumsum`, `np.cumprod` | Running totals/products | $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$ |
| `np.lib.stride_tricks.sliding_window_view` | Sliding windows without copies | Patch extraction for ViT |
| `np.sum/mean/max(axis=...)` | Reductions along specific axes | Per-channel normalization |
| `np.searchsorted` | Binary search into sorted arrays | Timestep binning |
| `np.digitize` | Bin continuous values | Schedule discretization |
| `np.bincount` | Count occurrences | Label histograms |

### The cumulative product pattern

In DDPM, the noise schedule is defined by $\beta_t$, and the key derived quantity is:

$$
\bar{\alpha}_t = \prod_{s=1}^{t} (1 - \beta_s)
$$

This is a single `np.cumprod` call — no loop needed.

In [ ]:
# --- Worked Example: Cumulative product of alphas (diffusion!) ---
# In DDPM, we define:
#   beta_t: noise schedule
#   alpha_t = 1 - beta_t
#   alpha_bar_t = prod_{s=1}^{t} alpha_s  (cumulative product)

T_diff = 1000
betas = np.linspace(1e-4, 0.02, T_diff, dtype=np.float64)  # shape (T,)
alphas = 1.0 - betas                                        # shape (T,)
alpha_bar = np.cumprod(alphas)                               # shape (T,)

print(f"betas range: [{betas[0]:.4f}, {betas[-1]:.4f}]")
print(f"alpha_bar[0]   = {alpha_bar[0]:.6f}  (close to 1: minimal noise)")
print(f"alpha_bar[-1]  = {alpha_bar[-1]:.6f}  (close to 0: mostly noise)")
print(f"alpha_bar[500] = {alpha_bar[500]:.6f}  (midway)")

In [ ]:
# --- Worked Example: Moving average with sliding_window_view ---

from numpy.lib.stride_tricks import sliding_window_view

signal = rng.standard_normal(100)  # shape (100,)
window_size = 5

# Create overlapping windows -- returns a VIEW, no data copied!
windows = sliding_window_view(signal, window_size)  # shape (96, 5)
print(f"Windows shape: {windows.shape}")
print(f"Is a view (shares memory): {np.shares_memory(windows, signal)}")

moving_avg = windows.mean(axis=1)  # shape (96,)
print(f"Moving average shape: {moving_avg.shape}")
print(f"First few values: {moving_avg[:5]}")

In [ ]:
# --- Worked Example: Vectorized histogram ---

data_hist = rng.standard_normal(10000)  # shape (10000,)
bin_edges = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=np.float64)

# Method 1: np.digitize + np.bincount
bin_indices = np.digitize(data_hist, bin_edges)  # shape (10000,) values in [0, len(edges)]
counts = np.bincount(bin_indices, minlength=len(bin_edges) + 1)  # shape (8,)
print("Manual histogram (digitize+bincount):")
for i in range(len(bin_edges) - 1):
    print(f"  [{bin_edges[i]:+.0f}, {bin_edges[i+1]:+.0f}): {counts[i+1]}")

# Method 2: np.histogram (built-in)
counts2, _ = np.histogram(data_hist, bins=bin_edges)
print(f"\nnp.histogram counts: {counts2}")

### Exercise 0.5

1. **Loop to vectorized: row norms.** Replace the loop with a single vectorized expression.
   ```python
   X = rng.standard_normal((1000, 50))
   norms = np.zeros(1000)
   for i in range(1000):
       norms[i] = np.sqrt(np.sum(X[i] ** 2))
   ```

2. **Loop to vectorized: cumulative max.** Compute the running maximum of a 1-D array without a loop. (Hint: `np.maximum.accumulate`.)

3. **Loop to vectorized: pairwise absolute differences.** Given `a` of shape `(N,)`, compute the `(N, N)` matrix `|a_i - a_j|` without loops.

4. **Cumprod from cumsum + exp/log.** Implement cumulative product using only `np.cumsum`, `np.log`, and `np.exp`. Compare numerical precision against `np.cumprod` on the alpha schedule.

5. **Timed pairwise cosine similarity.** For `X` of shape `(500, 128)`, compute the `(500, 500)` cosine similarity matrix. Time both a loop version and a vectorized version.

In [ ]:
# YOUR CODE HERE — Exercise 0.5

# 1. Vectorize row norms (replace the loop):
X_norm = rng.standard_normal((1000, 50))
# norms_vec = ???  # Should match: np.sqrt(np.sum(X_norm[i]**2)) for each i

# 2. Cumulative max without a loop. Hint: np.maximum.accumulate
arr_cm = rng.standard_normal(20)
# cummax = ???

# 3. Pairwise |a_i - a_j| matrix without loops.
a_pw = rng.standard_normal(100)
# pairwise_abs = ???  # shape (100, 100)

# 4. Implement cumprod via cumsum + exp/log (only valid for positive arrays).
def cumprod_via_cumsum(arr: np.ndarray) -> np.ndarray:
    """Cumulative product using log-space: exp(cumsum(log(arr)))."""
    pass

# 5. Timed pairwise cosine similarity for X (500, 128).
#    Write a vectorized version and compare speed to a loop.
X_cos = rng.standard_normal((500, 128))


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# 1. Row norms -- vectorized
X_norm = rng.standard_normal((1000, 50))  # shape (1000, 50)

# Loop version (slow)
norms_loop = np.zeros(1000)
for i in range(1000):
    norms_loop[i] = np.sqrt(np.sum(X_norm[i] ** 2))

# Vectorized version
norms_vec = np.sqrt(np.sum(X_norm ** 2, axis=1))  # shape (1000,)
# Alternative: np.linalg.norm(X_norm, axis=1)

assert np.allclose(norms_loop, norms_vec)
print("1. Row norms match.")

# 2. Cumulative max
arr_cm = rng.standard_normal(20)  # shape (20,)
cummax = np.maximum.accumulate(arr_cm)  # shape (20,)
print(f"\n2. Cumulative max:")
print(f"   Input:  {arr_cm[:8]}")
print(f"   CumMax: {cummax[:8]}")

# Verify
cummax_loop = np.zeros_like(arr_cm)
cummax_loop[0] = arr_cm[0]
for i in range(1, len(arr_cm)):
    cummax_loop[i] = max(cummax_loop[i-1], arr_cm[i])
assert np.allclose(cummax, cummax_loop)

# 3. Pairwise absolute differences
a_pw = rng.standard_normal(100)  # shape (100,)
pairwise_abs = np.abs(a_pw[:, None] - a_pw[None, :])  # shape (100, 100)
print(f"\n3. Pairwise abs diff shape: {pairwise_abs.shape}")
assert pairwise_abs[10, 20] == abs(a_pw[10] - a_pw[20])
print("   Verified.")

# 4. Cumprod via cumsum + exp/log
def cumprod_via_cumsum(arr: np.ndarray) -> np.ndarray:
    """Cumulative product using log-space: exp(cumsum(log(arr))).
    
    Only valid for strictly positive arrays.
    """
    return np.exp(np.cumsum(np.log(arr)))


# Test on alpha schedule (all values in (0,1), so log is valid)
alpha_bar_logspace = cumprod_via_cumsum(alphas)  # shape (T,)
max_err = np.max(np.abs(alpha_bar_logspace - alpha_bar))
print(f"\n4. Max error (log-space cumprod vs np.cumprod): {max_err:.2e}")
assert np.allclose(alpha_bar_logspace, alpha_bar, atol=1e-10)

# 5. Timed pairwise cosine similarity
X_cos = rng.standard_normal((500, 128))  # shape (500, 128)

# Loop version
t0 = time.perf_counter()
norms_x = np.linalg.norm(X_cos, axis=1)  # precompute for fair comparison
cos_loop = np.zeros((500, 500))
for i in range(500):
    for j in range(500):
        cos_loop[i, j] = np.dot(X_cos[i], X_cos[j]) / (norms_x[i] * norms_x[j])
t_loop = time.perf_counter() - t0

# Vectorized version
t0 = time.perf_counter()
X_normed = X_cos / np.linalg.norm(X_cos, axis=1, keepdims=True)  # shape (500, 128)
cos_vec = X_normed @ X_normed.T  # shape (500, 500)
t_vec = time.perf_counter() - t0

print(f"\n5. Cosine similarity:")
print(f"   Loop version:       {t_loop*1000:.1f} ms")
print(f"   Vectorized version: {t_vec*1000:.1f} ms")
print(f"   Speedup: {t_loop/t_vec:.0f}x")
assert np.allclose(cos_loop, cos_vec, atol=1e-10)

---
## 0.6 — Random Number Generation

**Why this matters for diffusion:** Diffusion models are fundamentally about noise — adding it (forward process), predicting it (training), and removing it (sampling). Every step requires controlled random number generation. Reproducibility via seeds is essential for debugging.

### Modern API: `np.random.default_rng(seed)`

The legacy `np.random.seed()` / `np.random.randn()` API uses global state. The modern `Generator` API is:
- **Explicit**: each `Generator` has its own state
- **Reproducible**: same seed → same sequence
- **Thread-safe**: no global state to corrupt

### The Reparameterization Trick

One of the most important ideas in generative modeling. We want to sample $z \sim \mathcal{N}(\mu, \sigma^2)$, but sampling is not differentiable w.r.t. $\mu$ and $\sigma$.

**Solution:** Reparameterize as a deterministic function of external noise:

$$
\varepsilon \sim \mathcal{N}(0, I) \qquad z = \mu + \sigma \cdot \varepsilon
$$

Now gradients flow through $\mu$ and $\sigma$ while $\varepsilon$ is treated as a fixed input.

**In diffusion**, the forward process uses exactly this:

$$
x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)
$$

In [ ]:
# --- Worked Example: Reproducible sampling ---

rng1 = np.random.default_rng(seed=123)
rng2 = np.random.default_rng(seed=123)

samples1 = rng1.standard_normal((3, 4))  # shape (3, 4)
samples2 = rng2.standard_normal((3, 4))  # shape (3, 4)

print("Same seed -> identical samples:")
print(f"  samples match: {np.array_equal(samples1, samples2)}")

# Different seed -> different samples
rng3 = np.random.default_rng(seed=456)
samples3 = rng3.standard_normal((3, 4))  # shape (3, 4)
print(f"  different seed matches: {np.array_equal(samples1, samples3)}")

In [ ]:
# --- Worked Example: Reparameterized Gaussian sampling ---

def sample_gaussian_reparam(
    mu: np.ndarray, sigma: np.ndarray, rng: np.random.Generator
) -> np.ndarray:
    """Sample z ~ N(mu, sigma^2) via the reparameterization trick.
    
    z = mu + sigma * eps, where eps ~ N(0, 1).
    Gradients flow through mu and sigma (not through the sampling operation).
    """
    eps = rng.standard_normal(mu.shape)  # shape same as mu
    return mu + sigma * eps  # shape same as mu


# Demo: sample from N(5, 2^2)
rng_demo = np.random.default_rng(42)
mu_demo = np.full((10000,), 5.0)    # shape (10000,)
sigma_demo = np.full((10000,), 2.0) # shape (10000,)
z_demo = sample_gaussian_reparam(mu_demo, sigma_demo, rng_demo)  # shape (10000,)

print(f"Sampled mean:  {z_demo.mean():.3f}  (expected: 5.0)")
print(f"Sampled std:   {z_demo.std():.3f}  (expected: 2.0)")

In [ ]:
# --- Worked Example: Visualize noise schedules ---
# (text-based, no matplotlib required)

T_sched = 1000
betas_linear = np.linspace(1e-4, 0.02, T_sched)  # shape (T,)
alphas_linear = 1.0 - betas_linear                 # shape (T,)
alpha_bar_linear = np.cumprod(alphas_linear)        # shape (T,)

# Cosine schedule (from Nichol & Dhariwal, 2021)
s_cos = 0.008
steps = np.arange(T_sched + 1, dtype=np.float64)  # shape (T+1,)
f = np.cos(((steps / T_sched) + s_cos) / (1 + s_cos) * (np.pi / 2)) ** 2  # shape (T+1,)
alpha_bar_cosine = f[1:] / f[0]  # shape (T,)
alpha_bar_cosine = np.clip(alpha_bar_cosine, 1e-8, 1.0)  # numerical safety

# Print a table of key timesteps
timesteps_to_show = [0, 100, 250, 500, 750, 999]
print(f"{'t':>5} | {'alpha_bar (linear)':>20} | {'alpha_bar (cosine)':>20}")
print("-" * 50)
for t in timesteps_to_show:
    print(f"{t:>5} | {alpha_bar_linear[t]:>20.6f} | {alpha_bar_cosine[t]:>20.6f}")

print("\nNote: The cosine schedule retains more signal at intermediate timesteps,")
print("producing better results in practice.")

In [ ]:
# --- Visualization: Noise schedules comparison ---

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: alpha_bar comparison
t_axis = np.arange(T_sched)
axes[0].plot(t_axis, alpha_bar_linear, label='Linear', linewidth=2)
axes[0].plot(t_axis, alpha_bar_cosine, label='Cosine', linewidth=2)
axes[0].set_xlabel('Timestep $t$')
axes[0].set_ylabel(r'$\bar{\alpha}_t$')
axes[0].set_title(r'Signal retention $\bar{\alpha}_t$')
axes[0].legend()

# Plot 2: sqrt(1 - alpha_bar) = noise scale
axes[1].plot(t_axis, np.sqrt(1 - alpha_bar_linear), label='Linear', linewidth=2)
axes[1].plot(t_axis, np.sqrt(1 - alpha_bar_cosine), label='Cosine', linewidth=2)
axes[1].set_xlabel('Timestep $t$')
axes[1].set_ylabel(r'$\sqrt{1 - \bar{\alpha}_t}$')
axes[1].set_title('Noise scale')
axes[1].legend()

# Plot 3: SNR in dB
snr_linear = alpha_bar_linear / (1 - alpha_bar_linear + 1e-8)
snr_cosine = alpha_bar_cosine / (1 - alpha_bar_cosine + 1e-8)
axes[2].plot(t_axis, 10 * np.log10(snr_linear + 1e-8), label='Linear', linewidth=2)
axes[2].plot(t_axis, 10 * np.log10(snr_cosine + 1e-8), label='Cosine', linewidth=2)
axes[2].set_xlabel('Timestep $t$')
axes[2].set_ylabel('SNR (dB)')
axes[2].set_title('Signal-to-Noise Ratio')
axes[2].legend()

plt.tight_layout()
plt.show()

print('Key insight: The cosine schedule preserves signal longer at intermediate')
print('timesteps, giving the model more useful training signal in that range.')

In [ ]:
# --- Visualization: Noise as distributions at different timesteps ---

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle('Effect of noise at different timesteps (1-D signal)', fontsize=14)

rng_viz = np.random.default_rng(42)
x_0_sample = 3.0  # A single clean "signal" value
n_samples_viz = 5000
timesteps_viz = [0, 50, 200, 500]

for col, t in enumerate(timesteps_viz):
    eps = rng_viz.standard_normal(n_samples_viz)
    
    # Linear schedule
    x_t_lin = np.sqrt(alpha_bar_linear[t]) * x_0_sample + np.sqrt(1 - alpha_bar_linear[t]) * eps
    axes[0, col].hist(x_t_lin, bins=50, density=True, alpha=0.7, color='C0')
    axes[0, col].set_title(f't={t}')
    axes[0, col].set_xlim(-5, 7)
    if col == 0:
        axes[0, col].set_ylabel('Linear')
    
    # Cosine schedule
    x_t_cos = np.sqrt(alpha_bar_cosine[t]) * x_0_sample + np.sqrt(1 - alpha_bar_cosine[t]) * eps
    axes[1, col].hist(x_t_cos, bins=50, density=True, alpha=0.7, color='C1')
    axes[1, col].set_xlim(-5, 7)
    if col == 0:
        axes[1, col].set_ylabel('Cosine')

plt.tight_layout()
plt.show()

print('As t increases, the distribution spreads from the clean value toward N(0, 1).')
print('The cosine schedule keeps the distribution tighter (more signal) at the same t.')

### Exercise 0.6

1. **Reparameterization for diagonal Gaussian.** Write a function `sample_diag_gaussian(mu, log_var, rng)` that samples from `N(mu, diag(exp(log_var)))`. Use `log_var` instead of `sigma` (common in VAE code because it is numerically stabler and unconstrained). Verify mean and variance on 100k samples.

2. **Mixture of Gaussians.** Sample 10,000 points from a mixture of 3 Gaussians with means `[-3, 0, 5]`, stds `[0.5, 1.0, 0.3]`, and weights `[0.3, 0.5, 0.2]`. Use only `rng.choice` and the reparameterization trick (no scipy).

3. **Spatial Gaussian noise for diffusion.** Given a batch of "images" of shape `(B, C, H, W) = (4, 3, 32, 32)`, generate noise `eps` of the same shape, then compute `x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps` for `t=500` using the linear schedule defined above.

In [ ]:
# YOUR CODE HERE — Exercise 0.6

# 1. Reparameterized diagonal Gaussian.
#    Sample z ~ N(mu, diag(exp(log_var))) using z = mu + exp(0.5*log_var) * eps.
#    Using log_var instead of sigma is standard in VAE code (numerically stabler).
def sample_diag_gaussian(
    mu: np.ndarray, log_var: np.ndarray, rng: np.random.Generator
) -> np.ndarray:
    """Sample from N(mu, diag(exp(log_var))) via reparameterization."""
    pass

# 2. Mixture of Gaussians: sample 10k points from 3 components.
#    means=[-3, 0, 5], stds=[0.5, 1.0, 0.3], weights=[0.3, 0.5, 0.2].
#    Use rng.choice for component selection + reparam trick.

# 3. Diffusion forward process: given x_0 of shape (B, C, H, W),
#    compute x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1-alpha_bar_t) * eps
#    for t=500 using the linear schedule.


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# 1. Diagonal Gaussian with log_var
def sample_diag_gaussian(
    mu: np.ndarray, log_var: np.ndarray, rng: np.random.Generator
) -> np.ndarray:
    """Sample from N(mu, diag(exp(log_var))) via reparameterization.
    
    sigma = exp(0.5 * log_var), then z = mu + sigma * eps.
    """
    sigma = np.exp(0.5 * log_var)  # shape same as mu
    eps = rng.standard_normal(mu.shape)  # shape same as mu
    return mu + sigma * eps  # shape same as mu


rng_ex6 = np.random.default_rng(42)
mu_test = np.array([2.0, -1.0, 0.5])            # shape (3,)
log_var_test = np.array([0.0, np.log(4.0), np.log(0.25)])  # variances: 1, 4, 0.25

n_samples = 100_000
samples_dg = np.array(
    [sample_diag_gaussian(mu_test, log_var_test, rng_ex6) for _ in range(n_samples)]
)  # shape (n_samples, 3)

print("1. Diagonal Gaussian verification:")
print(f"   Empirical means: {samples_dg.mean(axis=0)}  (expected: {mu_test})")
print(f"   Empirical vars:  {samples_dg.var(axis=0)}  (expected: {np.exp(log_var_test)})")

# 2. Mixture of Gaussians
rng_mix = np.random.default_rng(42)
n_mix = 10_000
means_mix = np.array([-3.0, 0.0, 5.0])    # shape (3,)
stds_mix = np.array([0.5, 1.0, 0.3])      # shape (3,)
weights_mix = np.array([0.3, 0.5, 0.2])   # shape (3,)

# Step 1: choose component for each sample
components = rng_mix.choice(3, size=n_mix, p=weights_mix)  # shape (n_mix,)

# Step 2: reparameterized sample from chosen component
eps_mix = rng_mix.standard_normal(n_mix)  # shape (n_mix,)
samples_mix = means_mix[components] + stds_mix[components] * eps_mix  # shape (n_mix,)

print(f"\n2. Mixture of Gaussians:")
print(f"   Overall mean: {samples_mix.mean():.3f}  (expected: {(weights_mix * means_mix).sum():.3f})")
# Check component proportions
for k in range(3):
    frac = (components == k).mean()
    print(f"   Component {k}: {frac:.3f} (expected: {weights_mix[k]:.3f})")

# 3. Spatial Gaussian noise for diffusion forward process
rng_diff = np.random.default_rng(42)
B_d, C_d, H_d, W_d = 4, 3, 32, 32
x_0 = rng_diff.standard_normal((B_d, C_d, H_d, W_d)).astype(np.float32)  # shape (B, C, H, W)

t_step = 500
ab_t = alpha_bar_linear[t_step]  # scalar

eps_diff = rng_diff.standard_normal(x_0.shape).astype(np.float32)  # shape (B, C, H, W)
x_t = (np.sqrt(ab_t).astype(np.float32) * x_0
       + np.sqrt(1.0 - ab_t).astype(np.float32) * eps_diff)  # shape (B, C, H, W)

print(f"\n3. Diffusion forward process:")
print(f"   x_0 shape: {x_0.shape}, x_t shape: {x_t.shape}")
print(f"   alpha_bar[{t_step}] = {ab_t:.6f}")
print(f"   sqrt(alpha_bar) = {np.sqrt(ab_t):.4f} (signal scale)")
print(f"   sqrt(1-alpha_bar) = {np.sqrt(1-ab_t):.4f} (noise scale)")
print(f"   x_0 std: {x_0.std():.4f}, x_t std: {x_t.std():.4f}")

---
## Comprehensive Challenges

Ten exercises that integrate everything from this module. Each has a solution cell immediately following.

### Challenge 1: Batch Normalization from Scratch

Implement batch normalization for a tensor of shape `(B, C, H, W)`:

```
mu_c = mean over (B, H, W) for each channel
var_c = var over (B, H, W) for each channel
x_hat = (x - mu_c) / sqrt(var_c + eps)
out = gamma * x_hat + beta
```

where `gamma` and `beta` are learnable per-channel parameters of shape `(C,)`.

In [ ]:
# YOUR CODE HERE — Challenge 1

def batch_norm(
    x: np.ndarray,
    gamma: np.ndarray,
    beta: np.ndarray,
    eps: float = 1e-5
) -> np.ndarray:
    """Batch normalization for (B, C, H, W) tensors.
    
    Steps:
    1. Compute per-channel mean and variance over (B, H, W)
    2. Normalize: x_hat = (x - mu) / sqrt(var + eps)
    3. Scale and shift: out = gamma * x_hat + beta
    
    Hint: use keepdims=True and reshape gamma/beta to (1, C, 1, 1).
    
    Expected: per-channel mean ≈ beta, std ≈ gamma after normalization.
    """
    pass


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def batch_norm(
    x: np.ndarray,
    gamma: np.ndarray,
    beta: np.ndarray,
    eps: float = 1e-5
) -> np.ndarray:
    """Batch normalization for (B, C, H, W) tensors.
    
    Args:
        x: Input tensor of shape (B, C, H, W).
        gamma: Scale parameter of shape (C,).
        beta: Shift parameter of shape (C,).
        eps: Small constant for numerical stability.
    
    Returns:
        Normalized tensor of shape (B, C, H, W).
    """
    # Compute stats over (B, H, W), keeping C
    mu = x.mean(axis=(0, 2, 3), keepdims=True)   # shape (1, C, 1, 1)
    var = x.var(axis=(0, 2, 3), keepdims=True)    # shape (1, C, 1, 1)
    
    x_hat = (x - mu) / np.sqrt(var + eps)         # shape (B, C, H, W)
    
    # Reshape gamma, beta for broadcasting: (C,) -> (1, C, 1, 1)
    gamma_r = gamma[None, :, None, None]           # shape (1, C, 1, 1)
    beta_r = beta[None, :, None, None]             # shape (1, C, 1, 1)
    
    return gamma_r * x_hat + beta_r                # shape (B, C, H, W)


# Test
rng_bn = np.random.default_rng(42)
x_bn = rng_bn.standard_normal((8, 3, 16, 16))  # shape (8, 3, 16, 16)
gamma_bn = np.ones(3)   # shape (3,)
beta_bn = np.zeros(3)   # shape (3,)

out_bn = batch_norm(x_bn, gamma_bn, beta_bn)  # shape (8, 3, 16, 16)
print(f"Output shape: {out_bn.shape}")

# Per-channel mean should be ~0, std ~1
for c in range(3):
    ch_mean = out_bn[:, c, :, :].mean()
    ch_std = out_bn[:, c, :, :].std()
    print(f"  Channel {c}: mean={ch_mean:.6f}, std={ch_std:.6f}")

# Test with non-trivial gamma, beta
gamma_bn2 = np.array([2.0, 0.5, 3.0])
beta_bn2 = np.array([1.0, -1.0, 0.0])
out_bn2 = batch_norm(x_bn, gamma_bn2, beta_bn2)
for c in range(3):
    ch_mean = out_bn2[:, c, :, :].mean()
    ch_std = out_bn2[:, c, :, :].std()
    print(f"  Channel {c} (scaled): mean={ch_mean:.4f} (exp ~{beta_bn2[c]:.1f}), std={ch_std:.4f} (exp ~{gamma_bn2[c]:.1f})")

### Challenge 2: Numerically Stable Softmax

Implement softmax along the last axis with the max-subtraction trick for numerical stability. Verify it handles extreme values without NaN/Inf.

In [ ]:
# YOUR CODE HERE — Challenge 2

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax along the given axis.
    
    Hint: subtract x.max(axis, keepdims=True) before exp to prevent overflow.
    Expected: rows sum to 1.0, no NaN/Inf even for extreme inputs like [1000, 1001, 1002].
    """
    pass


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax along the given axis.
    
    Subtracts the max before exponentiation to prevent overflow.
    """
    x_max = x.max(axis=axis, keepdims=True)  # shape: x with axis kept
    e_x = np.exp(x - x_max)                  # shape: same as x
    return e_x / e_x.sum(axis=axis, keepdims=True)  # shape: same as x


# Test: normal values
rng_sm = np.random.default_rng(42)
logits = rng_sm.standard_normal((4, 10))  # shape (4, 10)
probs = softmax(logits)  # shape (4, 10)
print(f"Sum per row (should be 1.0): {probs.sum(axis=1)}")
print(f"All positive: {(probs > 0).all()}")

# Test: extreme values (would cause overflow without max trick)
logits_extreme = np.array([[1000.0, 1001.0, 1002.0],
                           [-1000.0, -999.0, -998.0]])  # shape (2, 3)
probs_extreme = softmax(logits_extreme)  # shape (2, 3)
print(f"\nExtreme logits softmax:\n{probs_extreme}")
print(f"No NaN: {not np.any(np.isnan(probs_extreme))}")
print(f"No Inf: {not np.any(np.isinf(probs_extreme))}")
print(f"Sums: {probs_extreme.sum(axis=1)}")

# Test: 3-D tensor
logits_3d = rng_sm.standard_normal((2, 3, 5))  # shape (2, 3, 5)
probs_3d = softmax(logits_3d, axis=-1)  # shape (2, 3, 5)
print(f"\n3-D softmax sums: {probs_3d.sum(axis=-1)}")

### Challenge 3: KNN — Pairwise Distances + Top-k Vectorized

Implement k-nearest neighbors classification:
1. Compute pairwise L2 distances between query points and training points.
2. Find the k nearest neighbors using `np.argpartition`.
3. Return the majority label.

In [ ]:
# YOUR CODE HERE — Challenge 3

def knn_predict(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_query: np.ndarray,
    k: int = 5
) -> np.ndarray:
    """K-nearest neighbors classification, fully vectorized.
    
    Steps:
    1. Compute pairwise L2 distances (Q, N) using ||q-x||^2 = ||q||^2 + ||x||^2 - 2q·x
    2. Find k nearest using np.argpartition (O(N) per query)
    3. Majority vote with np.bincount
    
    Expected: points near cluster centers should be classified correctly.
    """
    pass


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def knn_predict(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_query: np.ndarray,
    k: int = 5
) -> np.ndarray:
    """K-nearest neighbors classification, fully vectorized.
    
    Args:
        X_train: Training features, shape (N, D).
        y_train: Training labels, shape (N,), integer class labels.
        X_query: Query features, shape (Q, D).
        k: Number of neighbors.
    
    Returns:
        Predicted labels, shape (Q,).
    """
    # Pairwise L2 distances: (Q, N)
    # ||q - x||^2 = ||q||^2 + ||x||^2 - 2 q.x  (more memory-efficient than broadcasting diff)
    q_sq = np.sum(X_query ** 2, axis=1, keepdims=True)  # shape (Q, 1)
    x_sq = np.sum(X_train ** 2, axis=1, keepdims=True)  # shape (N, 1)
    dist_sq = q_sq + x_sq.T - 2.0 * X_query @ X_train.T  # shape (Q, N)
    dist_sq = np.maximum(dist_sq, 0.0)  # numerical safety
    
    # Top-k nearest (smallest distances) using argpartition
    knn_idx = np.argpartition(dist_sq, k, axis=1)[:, :k]  # shape (Q, k)
    
    # Gather labels of k nearest neighbors
    knn_labels = y_train[knn_idx]  # shape (Q, k)
    
    # Majority vote per query
    n_classes = y_train.max() + 1
    predictions = np.zeros(X_query.shape[0], dtype=y_train.dtype)  # shape (Q,)
    for q in range(X_query.shape[0]):
        counts = np.bincount(knn_labels[q], minlength=n_classes)  # shape (n_classes,)
        predictions[q] = counts.argmax()
    
    return predictions


# Test with synthetic data
rng_knn = np.random.default_rng(42)
N_tr, D_knn, n_classes_knn = 200, 2, 3

# Generate clustered data
centers = np.array([[0, 0], [5, 5], [0, 5]], dtype=np.float64)  # shape (3, 2)
X_tr = np.vstack([
    rng_knn.standard_normal((N_tr // n_classes_knn, D_knn)) + centers[c]
    for c in range(n_classes_knn)
])  # shape (N_tr, 2) -- approximately, due to integer division
y_tr = np.repeat(np.arange(n_classes_knn), N_tr // n_classes_knn)  # shape (N_tr,)

# Query near each center
X_q = centers + rng_knn.standard_normal((3, 2)) * 0.1  # shape (3, 2)
preds = knn_predict(X_tr, y_tr, X_q, k=5)  # shape (3,)
print(f"Query points near centers 0, 1, 2")
print(f"Predictions: {preds}  (expected: [0, 1, 2])")
assert np.array_equal(preds, [0, 1, 2])

### Challenge 4: Image Patch Extraction with reshape/strides

Extract non-overlapping patches of size `(P, P)` from an image of shape `(H, W)` using only `reshape` and `transpose`. The result should be `(H//P, W//P, P, P)`.

In [ ]:
# YOUR CODE HERE — Challenge 4

def extract_patches(image: np.ndarray, patch_size: int) -> np.ndarray:
    """Extract non-overlapping patches from a 2-D image.
    
    Use reshape + transpose only (no loops, no sliding_window_view).
    (H, W) → (H//P, P, W//P, P) → transpose → (H//P, W//P, P, P)
    
    Expected: patches[0, 0] == image[:P, :P]
    """
    pass


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def extract_patches(image: np.ndarray, patch_size: int) -> np.ndarray:
    """Extract non-overlapping patches from a 2-D image.
    
    Args:
        image: Shape (H, W), where H and W are divisible by patch_size.
        patch_size: Side length of square patches.
    
    Returns:
        Patches of shape (H//P, W//P, P, P).
    """
    H, W = image.shape
    P = patch_size
    assert H % P == 0 and W % P == 0, "Dimensions must be divisible by patch_size"
    
    # Reshape: (H, W) -> (H//P, P, W//P, P) -> transpose to (H//P, W//P, P, P)
    patches = image.reshape(H // P, P, W // P, P)  # shape (H//P, P, W//P, P)
    patches = patches.transpose(0, 2, 1, 3)         # shape (H//P, W//P, P, P)
    return patches


# Test
H_p, W_p, P_size = 8, 12, 4
img = np.arange(H_p * W_p).reshape(H_p, W_p)  # shape (8, 12)
patches = extract_patches(img, P_size)  # shape (2, 3, 4, 4)
print(f"Image shape: {img.shape}")
print(f"Patches shape: {patches.shape}")
print(f"\nPatch [0, 0]:")
print(patches[0, 0])
print(f"\nTop-left 4x4 of image:")
print(img[:4, :4])
assert np.array_equal(patches[0, 0], img[:4, :4])
assert np.array_equal(patches[1, 2], img[4:8, 8:12])
print("\nAll patches verified.")

### Challenge 5: Convolution via im2col

**The idea:** Instead of sliding a kernel across an image with nested loops, we can:
1. **im2col:** Extract every overlapping patch into rows of a matrix (using `sliding_window_view`)
2. **Matrix multiply:** Multiply the patch matrix by the flattened kernel in one shot

This is how deep learning frameworks (cuDNN, MKL-DNN) actually implement convolution — it reduces to a highly optimized GEMM call.

**Implement** 2-D convolution (no padding, stride=1):
- Input: `image` shape `(H, W)`, `kernel` shape `(kH, kW)`
- Output: shape `(H-kH+1, W-kW+1)`
- Use `sliding_window_view` to extract patches → `(oH, oW, kH, kW)`
- Reshape to `(oH*oW, kH*kW)` and matmul with `kernel.reshape(-1)`
- Verify against manual `sum(image[r:r+kH, c:c+kW] * kernel)` at one position

In [ ]:
# YOUR CODE HERE — Challenge 5

from numpy.lib.stride_tricks import sliding_window_view as _swv

def conv2d_im2col(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """2-D convolution via im2col (no padding, stride=1).
    
    Steps:
    1. Extract overlapping patches with sliding_window_view → (oH, oW, kH, kW)
    2. Reshape to (oH*oW, kH*kW)
    3. Matrix-vector product with kernel.reshape(-1)
    4. Reshape result to (oH, oW)
    """
    pass

# Test with a Sobel-x kernel on a (10, 10) random image


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

from numpy.lib.stride_tricks import sliding_window_view as _swv


def conv2d_im2col(
    image: np.ndarray, kernel: np.ndarray
) -> np.ndarray:
    """2-D convolution via im2col (no padding, stride=1).
    
    Args:
        image: Shape (H, W).
        kernel: Shape (kH, kW).
    
    Returns:
        Output of shape (H-kH+1, W-kW+1).
    """
    kH, kW = kernel.shape
    # Extract all overlapping patches: shape (oH, oW, kH, kW)
    patches = _swv(image, (kH, kW))  # shape (oH, oW, kH, kW)
    oH, oW = patches.shape[0], patches.shape[1]
    
    # Flatten patches and kernel for matrix multiply
    col_matrix = patches.reshape(oH * oW, kH * kW)  # shape (oH*oW, kH*kW)
    kernel_flat = kernel.reshape(-1)                  # shape (kH*kW,)
    
    # Matrix-vector product
    output_flat = col_matrix @ kernel_flat  # shape (oH*oW,)
    return output_flat.reshape(oH, oW)     # shape (oH, oW)


# Test
rng_conv = np.random.default_rng(42)
img_conv = rng_conv.standard_normal((10, 10))  # shape (10, 10)
kernel_conv = np.array([[1, 0, -1],
                        [2, 0, -2],
                        [1, 0, -1]], dtype=np.float64)  # shape (3, 3) -- Sobel-x

out_conv = conv2d_im2col(img_conv, kernel_conv)  # shape (8, 8)
print(f"Input shape: {img_conv.shape}")
print(f"Kernel shape: {kernel_conv.shape}")
print(f"Output shape: {out_conv.shape}")

# Verify against manual convolution at one position
manual_val = np.sum(img_conv[2:5, 3:6] * kernel_conv)
assert np.isclose(out_conv[2, 3], manual_val)
print(f"out[2,3] = {out_conv[2,3]:.4f} (verified against manual: {manual_val:.4f})")

### Challenge 6: Cumulative Alpha Schedule

Given a beta schedule, compute all derived quantities used in DDPM:
- `alpha_t = 1 - beta_t`
- `alpha_bar_t = prod_{s=1}^{t} alpha_s`
- `sqrt_alpha_bar_t`
- `sqrt_one_minus_alpha_bar_t`
- `posterior_variance_t = beta_t * (1 - alpha_bar_{t-1}) / (1 - alpha_bar_t)`

In [ ]:
# YOUR CODE HERE — Challenge 6

def compute_diffusion_schedule(betas: np.ndarray) -> dict:
    """Compute all DDPM schedule quantities from a beta schedule.
    
    Return dict with keys:
    - 'alphas': 1 - betas
    - 'alpha_bar': cumulative product of alphas
    - 'sqrt_alpha_bar'
    - 'sqrt_one_minus_alpha_bar'
    - 'posterior_variance': beta_t * (1 - alpha_bar_{t-1}) / (1 - alpha_bar_t)
      (use alpha_bar_{-1} = 1.0 by convention)
    
    Expected: alpha_bar monotonically decreasing from ~1 to ~0.
    """
    pass


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def compute_diffusion_schedule(betas: np.ndarray) -> dict:
    """Compute all DDPM schedule quantities from a beta schedule.
    
    Args:
        betas: Noise schedule of shape (T,).
    
    Returns:
        Dictionary with all derived schedule arrays, each of shape (T,).
    """
    alphas = 1.0 - betas                         # shape (T,)
    alpha_bar = np.cumprod(alphas)                # shape (T,)
    sqrt_alpha_bar = np.sqrt(alpha_bar)           # shape (T,)
    sqrt_one_minus_ab = np.sqrt(1.0 - alpha_bar)  # shape (T,)
    
    # Posterior variance: beta_t * (1 - alpha_bar_{t-1}) / (1 - alpha_bar_t)
    # For t=0, alpha_bar_{t-1} = 1.0 by convention
    alpha_bar_prev = np.concatenate([[1.0], alpha_bar[:-1]])  # shape (T,)
    posterior_var = betas * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar)  # shape (T,)
    
    return {
        'betas': betas,
        'alphas': alphas,
        'alpha_bar': alpha_bar,
        'sqrt_alpha_bar': sqrt_alpha_bar,
        'sqrt_one_minus_alpha_bar': sqrt_one_minus_ab,
        'alpha_bar_prev': alpha_bar_prev,
        'posterior_variance': posterior_var,
    }


# Compute
T_ch6 = 1000
betas_ch6 = np.linspace(1e-4, 0.02, T_ch6, dtype=np.float64)  # shape (T,)
sched = compute_diffusion_schedule(betas_ch6)

print(f"Schedule computed for T={T_ch6}")
print(f"{'Quantity':<30} {'t=0':>12} {'t=250':>12} {'t=500':>12} {'t=999':>12}")
print("-" * 70)
for key in ['betas', 'alphas', 'alpha_bar', 'sqrt_alpha_bar',
            'sqrt_one_minus_alpha_bar', 'posterior_variance']:
    vals = sched[key]
    print(f"{key:<30} {vals[0]:>12.6f} {vals[250]:>12.6f} {vals[500]:>12.6f} {vals[999]:>12.6f}")

# Sanity checks
assert np.all(sched['alpha_bar'] > 0)
assert np.all(sched['alpha_bar'] < 1)
assert sched['alpha_bar'][0] > sched['alpha_bar'][-1]  # decreasing
print("\nAll sanity checks passed.")

### Challenge 7: Multinomial Sampling without Choice Loops

Given a batch of probability vectors `probs` of shape `(B, K)`, draw one sample per row (returning class indices). Use `rng.uniform` + cumulative sums — no Python loops over the batch.

In [ ]:
# YOUR CODE HERE — Challenge 7

def batched_multinomial(
    probs: np.ndarray, rng: np.random.Generator
) -> np.ndarray:
    """Sample one class per row from a batch of categorical distributions.
    
    No Python loops over the batch!
    Hint: use cumsum along axis=1, draw uniform u, find where u < cumprobs.
    
    Expected: empirical frequencies ≈ input probabilities over 10k samples.
    """
    pass


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def batched_multinomial(
    probs: np.ndarray, rng: np.random.Generator
) -> np.ndarray:
    """Sample one class per row from a batch of categorical distributions.
    
    Args:
        probs: Probability vectors, shape (B, K). Rows sum to 1.
        rng: Random number generator.
    
    Returns:
        Sampled class indices, shape (B,).
    """
    B, K = probs.shape
    cumprobs = np.cumsum(probs, axis=1)       # shape (B, K)
    u = rng.uniform(size=(B, 1))              # shape (B, 1)
    
    # For each row, find where u falls in the cumulative distribution
    # Compare u < cumprobs along axis=1, take first True
    samples = (u < cumprobs).argmax(axis=1)   # shape (B,)
    return samples


# Test
rng_mn = np.random.default_rng(42)
B_mn, K_mn = 10000, 4
probs_mn = np.array([0.1, 0.2, 0.3, 0.4])  # shape (4,)
probs_batch = np.tile(probs_mn, (B_mn, 1))  # shape (B, 4)

samples_mn = batched_multinomial(probs_batch, rng_mn)  # shape (B,)
empirical_probs = np.bincount(samples_mn, minlength=K_mn) / B_mn  # shape (4,)

print(f"Target probs:    {probs_mn}")
print(f"Empirical probs: {empirical_probs}")
print(f"Max deviation:   {np.max(np.abs(empirical_probs - probs_mn)):.4f}")
assert np.allclose(empirical_probs, probs_mn, atol=0.02)

### Challenge 8: Einsum Gauntlet (10 rapid-fire)

Write the einsum string for each operation and verify correctness.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

rng_eg = np.random.default_rng(42)

# Setup arrays
A_eg = rng_eg.standard_normal((4, 5))    # shape (4, 5)
B_eg = rng_eg.standard_normal((5, 3))    # shape (5, 3)
C_eg = rng_eg.standard_normal((4, 5, 3)) # shape (4, 5, 3)
v_eg = rng_eg.standard_normal((5,))      # shape (5,)
u_eg = rng_eg.standard_normal((4,))      # shape (4,)
S_eg = rng_eg.standard_normal((5, 5))    # shape (5, 5) -- square
X_batch_eg = rng_eg.standard_normal((3, 4, 5))  # shape (3, 4, 5)
Y_batch_eg = rng_eg.standard_normal((3, 5, 2))  # shape (3, 5, 2)

# Prepare a square sub-array for batch trace
X_sq = X_batch_eg[:, :4, :4].copy()  # shape (3, 4, 4)

tests = [
    ("1. MatMul A@B",
     'ij,jk->ik', (A_eg, B_eg), A_eg @ B_eg),
    ("2. Inner product v.v",
     'i,i->', (v_eg, v_eg), np.dot(v_eg, v_eg)),
    ("3. Outer product u x v",
     'i,j->ij', (u_eg, v_eg), np.outer(u_eg, v_eg)),
    ("4. Trace of S",
     'ii->', (S_eg,), np.trace(S_eg)),
    ("5. Batch matmul",
     'bij,bjk->bik', (X_batch_eg, Y_batch_eg), X_batch_eg @ Y_batch_eg),
    ("6. Element-wise then total sum",
     'ij,ij->', (A_eg, A_eg), (A_eg * A_eg).sum()),
    ("7. Row sums of A",
     'ij->i', (A_eg,), A_eg.sum(axis=1)),
    ("8. Column sums of A",
     'ij->j', (A_eg,), A_eg.sum(axis=0)),
    ("9. Contract 3-D along axis 1",
     'ijk,j->ik', (C_eg, v_eg), np.tensordot(C_eg, v_eg, axes=([1],[0]))),
    ("10. Batch trace",
     'bii->b', (X_sq,), np.array([np.trace(X_sq[b]) for b in range(3)])),
]

for desc, es, inputs, expected in tests:
    result = np.einsum(es, *inputs)
    match = np.allclose(result, expected)
    status = "OK" if match else "FAIL"
    print(f"{desc:<45} einsum('{es}') -> {status}")
    assert match, f"Failed on {desc}"

print("\nAll 10 einsum challenges passed.")

### Challenge 9: Memory Profiling

Predict the peak memory usage of each code snippet (in bytes), then verify.

In [ ]:
# YOUR CODE HERE — Challenge 9

# Predict peak memory usage in bytes for each snippet, then verify.

# Snippet 1: np.zeros((1000, 1000), dtype=np.float64)
#   Prediction: ??? bytes

# Snippet 2: np.zeros((256, 3, 32, 32), dtype=np.float32)
#   Prediction: ??? bytes

# Snippet 3: Pairwise distance with intermediate diff (N,M,D) float64
#   N, M, D = 500, 400, 128
#   Peak intermediate: ??? bytes

# Snippet 4: Does a view allocate memory? Does a copy?

# Snippet 5: Does (N,1) + (1,M) allocate an (N,M) result?


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# Snippet 1: Create float64 array of shape (1000, 1000)
# Prediction: 1000 * 1000 * 8 = 8,000,000 bytes = 8 MB
arr1 = np.zeros((1000, 1000), dtype=np.float64)
print(f"Snippet 1: {arr1.nbytes:,} bytes (predicted: 8,000,000)")
assert arr1.nbytes == 8_000_000

# Snippet 2: float32 array of shape (256, 3, 32, 32) -- batch of images
# Prediction: 256 * 3 * 32 * 32 * 4 = 3,145,728 bytes ~= 3 MB
arr2 = np.zeros((256, 3, 32, 32), dtype=np.float32)
print(f"Snippet 2: {arr2.nbytes:,} bytes (predicted: 3,145,728)")
assert arr2.nbytes == 3_145_728

# Snippet 3: Pairwise distance computation peak memory
# X (N,D), Y (M,D). Intermediate diff (N,M,D) dominates.
N_mem, M_mem, D_mem = 500, 400, 128
# Peak intermediate: N*M*D * 8 bytes (float64)
peak_intermediate = N_mem * M_mem * D_mem * 8
print(f"\nSnippet 3: Pairwise distance peak intermediate")
print(f"  diff array (N,M,D) = ({N_mem},{M_mem},{D_mem})")
print(f"  Size: {peak_intermediate:,} bytes = {peak_intermediate / 1e6:.1f} MB")

# Snippet 4: View vs copy memory impact
big = np.zeros((1000, 1000), dtype=np.float64)  # 8 MB
view_slice = big[::2, ::2]  # VIEW -- no additional memory
copy_slice = big[::2, ::2].copy()  # COPY -- 500*500*8 = 2 MB additional
print(f"\nSnippet 4:")
print(f"  big:        {big.nbytes:,} bytes")
print(f"  view_slice: shares memory = {np.shares_memory(view_slice, big)} -> 0 extra bytes")
print(f"  copy_slice: {copy_slice.nbytes:,} bytes additional")

# Snippet 5: Broadcasting memory -- does (N,1) + (1,M) allocate (N,M)?
N_bc, M_bc = 1000, 1000
a_bc = np.zeros((N_bc, 1), dtype=np.float64)  # 8 KB
b_bc = np.zeros((1, M_bc), dtype=np.float64)  # 8 KB
c_bc = a_bc + b_bc  # Result IS (N,M) = 8 MB -- broadcasting allocates full output
print(f"\nSnippet 5:")
print(f"  a: {a_bc.nbytes:,} bytes, b: {b_bc.nbytes:,} bytes")
print(f"  a + b result: {c_bc.nbytes:,} bytes (full output allocated!)")

del arr1, arr2, big, view_slice, copy_slice, a_bc, b_bc, c_bc

### Challenge 10: Timed Vectorization Challenge

Replace the loop with a fully vectorized solution and measure the speedup. The task: for each row of a matrix, compute the index of the element closest to the row's median.

In [ ]:
# YOUR CODE HERE — Challenge 10

# For each row of X, find the index of the element closest to the row's median.
# Write a vectorized version and time it against the loop version.

rng_ch10 = np.random.default_rng(42)
X_ch10 = rng_ch10.standard_normal((5000, 200))

# Loop version (baseline — try to beat this by 10x+):
# for i in range(len(X)):
#     med = np.median(X[i])
#     result[i] = np.argmin(np.abs(X[i] - med))

# Vectorized version:
# Hint: np.median with axis=1 and keepdims=True, then np.argmin


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

rng_ch10 = np.random.default_rng(42)
X_ch10 = rng_ch10.standard_normal((5000, 200))  # shape (5000, 200)

# --- Loop version ---
t0 = time.perf_counter()
result_loop = np.zeros(X_ch10.shape[0], dtype=np.int64)  # shape (5000,)
for i in range(X_ch10.shape[0]):
    med = np.median(X_ch10[i])
    result_loop[i] = np.argmin(np.abs(X_ch10[i] - med))
t_loop_ch10 = time.perf_counter() - t0

# --- Vectorized version ---
t0 = time.perf_counter()
medians = np.median(X_ch10, axis=1, keepdims=True)  # shape (5000, 1)
result_vec = np.argmin(np.abs(X_ch10 - medians), axis=1)  # shape (5000,)
t_vec_ch10 = time.perf_counter() - t0

print(f"Loop version:       {t_loop_ch10*1000:.1f} ms")
print(f"Vectorized version: {t_vec_ch10*1000:.1f} ms")
print(f"Speedup: {t_loop_ch10/t_vec_ch10:.1f}x")

assert np.array_equal(result_loop, result_vec)
print("Results match.")

---

**Module 0 complete.** You now have the NumPy fluency needed to implement diffusion models from scratch. Key takeaways:

- Understand memory layout (strides, views vs copies) to avoid silent bugs and unnecessary allocations.
- Broadcasting eliminates loops and temporary arrays — master the three rules.
- Einsum is a universal tool for any tensor contraction; learn the notation once, use it everywhere.
- Always vectorize: the performance gap between loops and NumPy operations is typically 10-100x.
- Use the modern `Generator` API for reproducible random numbers.
- The reparameterization trick (`z = mu + sigma * eps`) is foundational to diffusion and VAE sampling.

Proceed to **Module 1** when ready.